In [1]:
import os
import datetime as dt
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import classification_report, confusion_matrix

def generate_dataset_pdf_report(csv_file_path: str) -> str:
    """
    Reads a prediction CSV file and saves a single-page evaluation PDF report 
    containing a normalized confusion matrix heatmap and classification report table.
    """
    if not os.path.exists(csv_file_path):
        raise FileNotFoundError(f"File not found: {csv_file_path}")

    # Extract filename and set output path
    file_name = os.path.basename(csv_file_path)
    results_dir = os.path.dirname(csv_file_path) or '.'
    
    timestamp = dt.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_pdf_name = f"evaluation_report_{file_name.replace('.csv', '')}_{timestamp}.pdf"
    pdf_output_path = os.path.join(results_dir, output_pdf_name)

    # Load data
    df = pd.read_csv(csv_file_path)
    y_true = df['Actual_Activity']
    y_pred = df['Predicted_Activity']

    labels = sorted(list(set(y_true).union(set(y_pred))))

    # Compute metrics
    cls_dict = classification_report(y_true, y_pred, output_dict=True)
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Normalize Matrix by Row (True Classes) into Percentages (0% - 100%)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    # Build PDF
    with PdfPages(pdf_output_path) as pdf:
        fig = plt.figure(figsize=(8.5, 11))  # A4 Portrait
        
        # Title Block
        plt.suptitle(
            f"Dataset: {file_name}\n"
            f"Overall Accuracy: {cls_dict['accuracy']:.4f}  |  Macro F1-Score: {cls_dict['macro avg']['f1-score']:.4f}",
            fontsize=12,
            fontweight='bold',
            y=0.96
        )

        # 1. Confusion Matrix Heatmap (Top Half)
        ax1 = fig.add_subplot(2, 1, 1)
        annot_labels = np.array([[f"{val:.1f}%" for val in row] for row in cm_normalized])

        sns.heatmap(
            cm_normalized,
            annot=annot_labels,
            fmt="",
            cmap='Blues',
            xticklabels=labels,
            yticklabels=labels,
            cbar=True,
            cbar_kws={'label': 'Percentage (%)'},
            vmin=0,
            vmax=100,
            ax=ax1
        )
        
        ax1.set_title('Normalized Matrix (Actual vs. Predicted %)', fontsize=11, fontweight='bold', pad=10)
        ax1.set_xlabel('Predicted Activity', fontsize=9, fontweight='bold')
        ax1.set_ylabel('Actual Activity', fontsize=9, fontweight='bold')
        plt.setp(ax1.get_xticklabels(), rotation=30, ha='right', fontsize=8)
        plt.setp(ax1.get_yticklabels(), rotation=0, fontsize=8)

        # 2. Classification Report Table (Bottom Half)
        ax2 = fig.add_subplot(2, 1, 2)
        ax2.axis('off')

        table_data = []
        headers = ['Class / Metric', 'Precision', 'Recall', 'F1-Score', 'Support']

        for key, value in cls_dict.items():
            if isinstance(value, dict):
                table_data.append([
                    key,
                    f"{value['precision']:.4f}",
                    f"{value['recall']:.4f}",
                    f"{value['f1-score']:.4f}",
                    int(value['support'])
                ])
            elif key == 'accuracy':
                table_data.append(['accuracy', '', '', f"{value:.4f}", len(y_true)])

        table = ax2.table(
            cellText=table_data,
            colLabels=headers,
            loc='center',
            cellLoc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.0, 1.4)

        # Highlight header row
        for (row_idx, col_idx), cell in table.get_celld().items():
            if row_idx == 0:
                cell.set_facecolor('#d3d3d3')
                cell.get_text().set_weight('bold')

        ax2.set_title('Classification Report', fontsize=11, fontweight='bold', pad=10)

        plt.tight_layout(rect=[0, 0.03, 1, 0.93])
        
        # Save figure directly to PDF file and suppress output display
        pdf.savefig(fig)
        plt.close(fig)

    print(f"Saved PDF to: {pdf_output_path}")
    return pdf_output_path


# Single line call:
generate_dataset_pdf_report('../dataset/results/pred_full_dataset_raw_features_2026-09-21_19-44-12_xgboost.csv')

Saved PDF to: ../dataset/results/evaluation_report_pred_full_dataset_raw_features_2026-09-21_19-44-12_xgboost_2026-09-22_14-17-28.pdf


'../dataset/results/evaluation_report_pred_full_dataset_raw_features_2026-09-21_19-44-12_xgboost_2026-09-22_14-17-28.pdf'